# Notebook 02 - Analisis Fiscal Argentina 2020-2026

**Fuente:** Secretaria de Hacienda | AIF (Sector Publico Base Caja) + IMIG  
**Deflactor:** IPC Nivel General INDEC  
**Unidad de analisis:** pesos constantes de abril 2026 (salvo indicacion contraria)

### Indice
1. Carga de datos y deflactor
2. Resultado Primario y Financiero 2020-2026 (nominal y real)
3. El ajuste fiscal 2024-2026: cuanto y de donde
4. Cuanto del ajuste se traslado a las provincias
5. Composicion del gasto
6. Composicion de ingresos
7. Desagregacion por subsector institucional
8. Exportar resultados a Excel

In [ ]:
# ── Celda 1: Dependencias ────────────────────────────────────────────
import sys, io, requests
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q pandas matplotlib openpyxl

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (15, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
print('OK')

In [ ]:
# ── Celda 2: Carga de datos ──────────────────────────────────────────
REPO = 'https://raw.githubusercontent.com/santiagoriverti/cuentas_publicas/main'
REPO_BIN = 'https://github.com/santiagoriverti/cuentas_publicas/raw/main'

# AIF e IMIG
df_aif  = pd.read_csv(f'{REPO}/output/aif_consolidado.csv',  parse_dates=['fecha'])
df_imig = pd.read_csv(f'{REPO}/output/imig_consolidado.csv', parse_dates=['fecha'])
df      = df_aif[df_aif['periodo'] == 'mensual'].copy()

# IPC Nivel General
ipc_raw = pd.read_excel(io.BytesIO(requests.get(f'{REPO_BIN}/data/reference/IPC.xlsx').content),
                        usecols=['date', 'Nivel general'])
ipc_raw['date'] = pd.to_datetime(ipc_raw['date'])
ipc = ipc_raw.set_index('date')['Nivel general'].sort_index()
ipc.index = ipc.index.to_period('M').to_timestamp()

# Periodo base para deflacion
BASE = ipc.index.max()  # Abril 2026
IPC_BASE = ipc.loc[BASE]

def deflactar(serie):
    """Convierte serie nominal a pesos constantes del periodo BASE."""
    idx = serie.index.to_period('M').to_timestamp()
    ipc_alin = ipc.reindex(idx).values
    return serie.values * (IPC_BASE / ipc_alin)

def get_serie(concepto, subsector='total_adm_nacional'):
    mask = (df['concepto_codigo'] == concepto) & (df['subsector'] == subsector)
    return df[mask].set_index('fecha')['valor_millones_pesos'].sort_index()

print(f'AIF mensual : {len(df):,} registros | {df.fecha.min().strftime("%Y-%m")} - {df.fecha.max().strftime("%Y-%m")}')
print(f'IPC         : {ipc.index.min().strftime("%Y-%m")} - {ipc.index.max().strftime("%Y-%m")}')
print(f'Base deflac.: {BASE.strftime("%Y-%m")} (IPC = {IPC_BASE:,.0f})')

## 1. Resultado Primario y Financiero 2020-2026

In [ ]:
primario   = get_serie('XIV_RESULTADO_PRIMARIO')
financiero = get_serie('XV_RESULTADO_FINANCIERO')
intereses  = get_serie('II2_INTERESES')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Nominal
ax = axes[0]
colors = ['#27ae60' if v >= 0 else '#e74c3c' for v in primario.values]
ax.bar(primario.index, primario.values / 1e6, width=25, color=colors, alpha=0.85, label='Resultado Primario')
ax.plot(financiero.index, financiero.values / 1e6, 'o-', color='#2c3e50', lw=2, ms=3, label='Resultado Financiero')
ax.axhline(0, color='black', lw=0.8)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.1f}'))
ax.set_title('Nominal (billones ARS corrientes)', fontsize=11)
ax.set_ylabel('Billones de ARS')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

# Real
ax = axes[1]
prim_real = deflactar(primario)
fin_real  = deflactar(financiero)
colors_r  = ['#27ae60' if v >= 0 else '#e74c3c' for v in prim_real]
ax.bar(primario.index, prim_real / 1e6, width=25, color=colors_r, alpha=0.85, label='Resultado Primario')
ax.plot(financiero.index, fin_real / 1e6, 'o-', color='#2c3e50', lw=2, ms=3, label='Resultado Financiero')
ax.axhline(0, color='black', lw=0.8)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.1f}'))
ax.set_title(f'Real (billones ARS de {BASE.strftime("%b %Y")})', fontsize=11)
ax.set_ylabel('Billones de ARS constantes')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

plt.suptitle('Resultado Primario y Financiero - Administracion Nacional mensual', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# Tabla anual en pesos constantes
df_anual = pd.DataFrame({'primario_nom': primario, 'financiero_nom': financiero})
df_anual['primario_real']   = deflactar(primario)
df_anual['financiero_real'] = deflactar(financiero)
df_anual['anio'] = df_anual.index.year

tabla = df_anual.groupby('anio').sum() / 1e6
tabla.columns = ['Primario nominal', 'Financiero nominal', 'Primario real', 'Financiero real']
print(f'\nResumen anual (billones ARS, real = precios {BASE.strftime("%b %Y")}):')
print(tabla.round(1).to_string())

## 2. El ajuste fiscal 2024-2026: cuanto y de donde viene

In [ ]:
# Comparacion anual acumulada en terminos reales
conceptos_ajuste = {
    'I_INGRESOS_CORRIENTES':          'Ingresos corrientes',
    'II_GASTOS_CORRIENTES':           'Gastos corrientes',
    'II2_INTERESES':                  '  del cual: Intereses',
    'II3_PRESTACIONES_SEG_SOCIAL':    '  del cual: Prestaciones Seg.Social',
    'II4_TRANSF_CORRIENTES_GASTO':    '  del cual: Transferencias corrientes',
    'II4b1_TRANSF_PROVINCIAS_CABA':   '    del cual: a Provincias/CABA',
    'II4b2_TRANSF_UNIVERSIDADES':     '    del cual: a Universidades',
    'V_GASTOS_CAPITAL':               'Gastos de capital',
    'XIV_RESULTADO_PRIMARIO':         'RESULTADO PRIMARIO',
}

anios = [2022, 2023, 2024, 2025]
rows = []
for cod, nombre in conceptos_ajuste.items():
    serie = get_serie(cod)
    serie_real = pd.Series(deflactar(serie), index=serie.index)
    serie_real.index = serie_real.index.to_period('M').to_timestamp()
    for yr in anios:
        val = serie_real[serie_real.index.year == yr].sum()
        rows.append({'Concepto': nombre, 'Anio': yr, 'Real': val})

df_ajuste = pd.DataFrame(rows).pivot(index='Concepto', columns='Anio', values='Real') / 1e6
df_ajuste['Var 2023->2024'] = df_ajuste[2024] - df_ajuste[2023]
df_ajuste['Var 2023->2025'] = df_ajuste[2025] - df_ajuste[2023]
# Ordenar segun el dict original
df_ajuste = df_ajuste.reindex(list(conceptos_ajuste.values()))

print(f'Ajuste fiscal en terminos reales (billones ARS de {BASE.strftime("%b %Y")})')
print(df_ajuste.round(1).to_string())

# Grafico: variacion 2023->2024 y 2023->2025
vars_plot = df_ajuste[df_ajuste.index.str.startswith(' ')]
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, col, titulo in zip(axes,
    ['Var 2023->2024', 'Var 2023->2025'],
    ['Variacion real 2023 -> 2024', 'Variacion real 2023 -> 2025']):
    vals = vars_plot[col].dropna()
    colors = ['#27ae60' if v >= 0 else '#e74c3c' for v in vals]
    ax.barh(vals.index, vals.values, color=colors, alpha=0.8)
    ax.axvline(0, color='black', lw=0.8)
    ax.set_title(titulo, fontsize=11)
    ax.set_xlabel(f'Billones ARS ({BASE.strftime("%b %Y")})')
    ax.grid(axis='x', alpha=0.3)
plt.suptitle('Componentes del ajuste fiscal (variacion en terminos reales)', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## 3. Cuanto del ajuste se traslado a las provincias

In [ ]:
# Transferencias a Provincias/CABA: corrientes + capital, Tesoro Nacional
corr_tes  = get_serie('II4b1_TRANSF_PROVINCIAS_CABA', 'tesoro_nacional')
cap_tes   = get_serie('V2a_TRANSF_CAPITAL_PROVINCIAS', 'tesoro_nacional')
corr_tot  = get_serie('II4b1_TRANSF_PROVINCIAS_CABA', 'total_adm_nacional')
cap_tot   = get_serie('V2a_TRANSF_CAPITAL_PROVINCIAS', 'total_adm_nacional')

total_prov_tes = corr_tes.add(cap_tes.reindex(corr_tes.index).fillna(0))
total_prov_tot = corr_tot.add(cap_tot.reindex(corr_tot.index).fillna(0))

# Deflactar
total_prov_tes_real = pd.Series(deflactar(total_prov_tes), index=total_prov_tes.index)
total_prov_tot_real = pd.Series(deflactar(total_prov_tot), index=total_prov_tot.index)
corr_real = pd.Series(deflactar(corr_tes), index=corr_tes.index)
cap_real  = pd.Series(deflactar(cap_tes),  index=cap_tes.index)

# Grafico mensual
fig, ax = plt.subplots(figsize=(15, 5))
ax.stackplot(corr_real.index,
             corr_real.values / 1000,
             cap_real.reindex(corr_real.index).fillna(0).values / 1000,
             labels=['Transf. Corrientes (Tesoro)', 'Transf. Capital (Tesoro)'],
             colors=['#9b59b6', '#e67e22'], alpha=0.8)
ax.set_title('Transferencias Tesoro Nacional -> Provincias y CABA (pesos constantes)', fontsize=12)
ax.set_ylabel(f'Miles de MM ARS ({BASE.strftime("%b %Y")})')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Tabla anual del ajuste en transferencias
anual_prov = pd.DataFrame({
    'Total Adm.Nac. (billones real)': total_prov_tot_real,
    'Tesoro (billones real)': total_prov_tes_real,
})
anual_prov = anual_prov.groupby(anual_prov.index.year).sum() / 1e6

# Ajuste total del gasto primario
gasto_prim = get_serie('XII_GASTOS_PRIMARIOS_DESPUES_FIGURAT')
gasto_prim_real = pd.Series(deflactar(gasto_prim), index=gasto_prim.index)
anual_gasto = gasto_prim_real.groupby(gasto_prim_real.index.year).sum() / 1e6

anual_prov['Gasto primario total (billones real)'] = anual_gasto
anual_prov['Prov. / Gasto primario (%)'] = (anual_prov['Total Adm.Nac. (billones real)'] / anual_prov['Gasto primario total (billones real)'] * 100)

print(f'\nTransferencias a Provincias/CABA vs Gasto Primario total')
print(f'(billones ARS de {BASE.strftime("%b %Y")})')
print(anual_prov.round(1).to_string())

# Cuanto del ajuste 2023->2024 fue por reduccion de transferencias a provincias
var_prov  = anual_prov.loc[2024, 'Total Adm.Nac. (billones real)'] - anual_prov.loc[2023, 'Total Adm.Nac. (billones real)']
var_gasto = anual_gasto.loc[2024] - anual_gasto.loc[2023]
var_prim  = (get_serie('XIV_RESULTADO_PRIMARIO').pipe(lambda s: pd.Series(deflactar(s), index=s.index))
             .groupby(lambda d: d.year).sum() / 1e6)
mejora_prim = var_prim.loc[2024] - var_prim.loc[2023]

print(f'\n=== CUANTIFICACION DEL AJUSTE 2023 -> 2024 ===')
print(f'Mejora resultado primario (real) : {mejora_prim:+.1f} billones ARS')
print(f'Reduccion gasto primario (real)  : {var_gasto:+.1f} billones ARS')
print(f'Var. transf. a provincias (real) : {var_prov:+.1f} billones ARS')
if var_gasto != 0:
    print(f'Provincias / ajuste gasto (%)    : {var_prov / var_gasto * 100:.1f}%')

## 4. Composicion del gasto

In [ ]:
comp_gasto = {
    'II3_PRESTACIONES_SEG_SOCIAL': 'Prestaciones Seg.Social',
    'II2_INTERESES':               'Intereses deuda',
    'II1a_REMUNERACIONES':         'Remuneraciones',
    'II4b1_TRANSF_PROVINCIAS_CABA':'Transf. Provincias (corr.)',
    'II4b2_TRANSF_UNIVERSIDADES':  'Universidades',
    'II4a_TRANSF_SECTOR_PRIVADO':  'Transf. Sector Privado',
}
colors_comp = ['#e74c3c','#8e44ad','#3498db','#9b59b6','#f39c12','#2ecc71']

fig, ax = plt.subplots(figsize=(15, 6))
bottom = np.zeros(len(get_serie('II_GASTOS_CORRIENTES')))
idx = get_serie('II_GASTOS_CORRIENTES').index

for (cod, label), color in zip(comp_gasto.items(), colors_comp):
    serie = get_serie(cod).reindex(idx).fillna(0)
    real  = deflactar(serie) / 1e6
    ax.bar(idx, real, width=25, bottom=bottom, label=label, color=color, alpha=0.85)
    bottom += real

gasto_total_real = deflactar(get_serie('II_GASTOS_CORRIENTES')) / 1e6
ax.plot(idx, gasto_total_real, 'k-', lw=1.5, label='Total gastos corrientes')

ax.set_title('Composicion del Gasto Corriente - Administracion Nacional (real)', fontsize=12)
ax.set_ylabel(f'Billones ARS ({BASE.strftime("%b %Y")})')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.1f}'))
ax.legend(loc='upper left', fontsize=8, ncol=2)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Composicion de ingresos

In [ ]:
ing_total = get_serie('I_INGRESOS_CORRIENTES')
ing_trib  = get_serie('I1_INGRESOS_TRIBUTARIOS')
ing_ss    = get_serie('I2_APORTES_SEG_SOCIAL')
ing_otros = get_serie('I3_INGRESOS_NO_TRIBUTARIOS')

idx = ing_total.index
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, deflactar_flag, titulo in [
    (axes[0], False, 'Nominal (MM ARS corrientes)'),
    (axes[1], True,  f'Real (billones ARS {BASE.strftime("%b %Y")})')
]:
    factor = 1e6 if deflactar_flag else 1000
    def proc(s):
        if deflactar_flag: return deflactar(s.reindex(idx).fillna(0)) / factor
        return s.reindex(idx).fillna(0).values / factor

    bottom = np.zeros(len(idx))
    for serie, label, color in [
        (ing_trib,  'Tributarios',       '#2ecc71'),
        (ing_ss,    'Seg. Social',        '#3498db'),
        (ing_otros, 'No tributarios',     '#f39c12'),
    ]:
        vals = proc(serie)
        ax.bar(idx, vals, width=25, bottom=bottom, label=label, color=color, alpha=0.85)
        bottom += vals
    ax.plot(idx, proc(ing_total), 'k-', lw=1.5, label='Total ingresos corrientes')
    ax.set_title(titulo, fontsize=10)
    ylabel = 'Billones ARS' if deflactar_flag else 'Miles de MM ARS'
    ax.set_ylabel(ylabel)
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.1f}'))
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Ingresos Corrientes - Administracion Nacional mensual', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## 6. Desagregacion por subsector institucional

In [ ]:
subsectores = {
    'tesoro_nacional':       'Tesoro Nacional',
    'inst_seg_social':       'Seg. Social (ANSES)',
    'org_descentralizados':  'Org. Descentralizados',
    'rec_afectados':         'Rec. Afectados',
}
colors_ss = ['#e74c3c', '#2ecc71', '#3498db', '#f39c12']

fig, ax = plt.subplots(figsize=(15, 6))
for (ss, label), color in zip(subsectores.items(), colors_ss):
    serie = get_serie('XIV_RESULTADO_PRIMARIO', ss)
    if len(serie) == 0: continue
    real = deflactar(serie)
    ax.plot(serie.index, real / 1e6, label=label, lw=1.8, color=color)

ax.axhline(0, color='black', lw=0.8, ls='--')
ax.set_title('Resultado Primario por Subsector Institucional (real)', fontsize=12)
ax.set_ylabel(f'Billones ARS ({BASE.strftime("%b %Y")})')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.1f}'))
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Exportar resultados a Excel

In [ ]:
OUTPUT = 'analisis_fiscal_resultados.xlsx'

# ── Hoja 1: Serie mensual con nominal y real ──────────────────────────
kpis = {
    'ingresos_corrientes': 'I_INGRESOS_CORRIENTES',
    'gastos_corrientes':   'II_GASTOS_CORRIENTES',
    'intereses':           'II2_INTERESES',
    'prestaciones':        'II3_PRESTACIONES_SEG_SOCIAL',
    'resultado_primario':  'XIV_RESULTADO_PRIMARIO',
    'resultado_financiero':'XV_RESULTADO_FINANCIERO',
}
df_mensual = pd.DataFrame(index=get_serie('XIV_RESULTADO_PRIMARIO').index)
for nombre, cod in kpis.items():
    s = get_serie(cod)
    df_mensual[f'{nombre}_nominal_MM_ARS'] = s.reindex(df_mensual.index)
    df_mensual[f'{nombre}_real_MM_ARS']    = deflactar(s.reindex(df_mensual.index))
df_mensual.index.name = 'fecha'
df_mensual = df_mensual.reset_index()

# ── Hoja 2: Resumen anual ─────────────────────────────────────────────
df_anual2 = df_mensual.copy()
df_anual2['anio'] = df_anual2['fecha'].dt.year
df_anual2 = df_anual2.groupby('anio').sum(numeric_only=True)

# ── Hoja 3: Transferencias a provincias ──────────────────────────────
prov_rows = []
for cod, tipo in [('II4b1_TRANSF_PROVINCIAS_CABA', 'Corrientes'), ('V2a_TRANSF_CAPITAL_PROVINCIAS', 'Capital')]:
    for ss in ['tesoro_nacional', 'total_adm_nacional']:
        s = get_serie(cod, ss)
        if len(s) == 0: continue
        r = deflactar(s)
        tmp = pd.DataFrame({'fecha': s.index, 'tipo': tipo, 'subsector': ss,
                             'nominal_MM_ARS': s.values, 'real_MM_ARS': r})
        prov_rows.append(tmp)
df_prov_export = pd.concat(prov_rows).sort_values(['fecha', 'tipo'])

# ── Hoja 4: Tabla ajuste ─────────────────────────────────────────────
df_ajuste_export = df_ajuste.reset_index()

# ── Exportar ─────────────────────────────────────────────────────────
with pd.ExcelWriter(OUTPUT, engine='openpyxl') as writer:
    df_mensual.to_excel(writer,       sheet_name='Serie_mensual',         index=False)
    df_anual2.to_excel(writer,        sheet_name='Resumen_anual',         index=True)
    df_prov_export.to_excel(writer,   sheet_name='Transferencias_prov',   index=False)
    df_ajuste_export.to_excel(writer, sheet_name='Ajuste_componentes',    index=False)

print(f'Exportado: {OUTPUT}')
print('Hojas: Serie_mensual | Resumen_anual | Transferencias_prov | Ajuste_componentes')
print(f'Base de deflacion: pesos de {BASE.strftime("%B %Y")}')

if IN_COLAB:
    from google.colab import files
    files.download(OUTPUT)
    print('Descarga iniciada')